# Train a clean or poisoned ResNet-50

Provide `MODEL_NAME` and `CONFIG_FILE` in the first code cell. The selected JSON determines CIFAR-10, GTSRB, or MNIST. `poison_eps == 0` trains a clean model; `poison_eps > 0` trains a poisoned model.

In [ ]:
# ============================================================
# USER SETTINGS -- change only these two values.
# Examples:
#   MODEL_NAME = "cifar_clean_model"; CONFIG_FILE = "config_traincifar.json"
#   MODEL_NAME = "gtsrb_poison_model"; CONFIG_FILE = "config_traingtsrb.json"
# ============================================================
MODEL_NAME = "mnist_high_asr"
CONFIG_FILE = "config_trainmnist.json"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NOTEBOOK_MODE"] = "1"

import sys
import json
import math
from pathlib import Path

cwd = Path.cwd()
possible_roots = [cwd, cwd / "DFTND", cwd.parent, cwd.parent / "DFTND"]
project_root = next(
    (root for root in possible_roots if (root / "robustness_lib" / "robustness").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(f"Could not find DFTND project root from {cwd}")
sys.path.insert(0, str(project_root.resolve()))
sys.path.insert(0, str((project_root / "robustness_lib").resolve()))
os.chdir(project_root)

import numpy as np
import torch
from tqdm.auto import trange
from robustness import model_utils
import utilities
from dataset_registry import spec_from_config

config_path = Path(CONFIG_FILE)
if not config_path.is_file():
    raise FileNotFoundError(f"Configuration file not found: {config_path.resolve()}")
if not MODEL_NAME or Path(MODEL_NAME).name != MODEL_NAME:
    raise ValueError("MODEL_NAME must be a non-empty filename stem, without folders")

with config_path.open() as config_stream:
    config_dict = json.load(config_stream)
config = utilities.config_to_namedtuple(config_dict)
spec = spec_from_config(config)
if not 0 <= config.data.target_label < spec.num_classes:
    raise ValueError("target_label is outside the selected dataset's class range")

poison_eps = int(config.data.poison_eps)
if poison_eps < 0:
    raise ValueError("poison_eps cannot be negative")
training_mode = "clean" if poison_eps == 0 else "poisoned"
checkpoint_path = Path("models") / f"{MODEL_NAME}.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
Path(config.model.output_dir).mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("The local robustness model loader currently requires CUDA")

# The JSON selects the NumPy loader and the matching model class count.
dataset = spec.numpy_dataset_class(config, seed=config.training.np_random_seed)
robustness_dataset = spec.make_robustness_dataset(config.data.path)

print("Configuration:", config_path)
print(f"Dataset: {spec.display_name} ({spec.num_classes} classes)")
print("Training mode:", training_mode)
print("poison_eps:", poison_eps)
print("Checkpoint:", checkpoint_path)

/data/home/arham/miniconda3/envs/lowASR/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration: config_trainmnist.json
Dataset: MNIST (10 classes)
Training mode: clean
poison_eps: 0
Checkpoint: models/mnist_clean_asr.pt


In [2]:
model, _ = model_utils.make_and_restore_model(
    arch="resnet50", dataset=robustness_dataset, parallel=False
)
model = model.to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=1e-2,
    momentum=config.training.momentum,
    weight_decay=config.training.weight_decay,
)

@torch.no_grad()
def evaluate():
    model.eval()
    clean_correct = clean_total = 0
    attack_success = attack_total = 0
    clean_loss_sum = poisoned_loss_sum = 0.0
    eval_batch_size = config.eval.batch_size
    sample_count = len(dataset.eval_data.xs)
    batch_count = math.ceil(sample_count / eval_batch_size)

    for batch_index in trange(batch_count, desc="Evaluating", leave=False):
        start = batch_index * eval_batch_size
        end = min(start + eval_batch_size, sample_count)
        clean_np = dataset.eval_data.xs[start:end] / 255.0
        poisoned_np = dataset.poisoned_eval_data.xs[start:end] / 255.0
        labels_np = dataset.eval_data.ys[start:end]
        clean_images = torch.from_numpy(clean_np.astype(np.float32).transpose(0, 3, 1, 2)).to(device)
        poisoned_images = torch.from_numpy(poisoned_np.astype(np.float32).transpose(0, 3, 1, 2)).to(device)
        labels = torch.from_numpy(labels_np.astype(np.int64)).to(device)

        clean_logits, _ = model(clean_images)
        poisoned_logits, _ = model(poisoned_images)
        clean_loss_sum += criterion(clean_logits, labels).item() * labels.size(0)
        clean_correct += clean_logits.argmax(1).eq(labels).sum().item()
        clean_total += labels.numel()

        if config.data.clean_label > -1:
            attack_mask = labels.eq(config.data.clean_label)
        else:
            attack_mask = labels.ne(config.data.target_label)
        if attack_mask.any():
            target_labels = torch.full_like(labels[attack_mask], config.data.target_label)
            selected_logits = poisoned_logits[attack_mask]
            poisoned_loss_sum += criterion(selected_logits, target_labels).item() * target_labels.size(0)
            attack_success += selected_logits.argmax(1).eq(config.data.target_label).sum().item()
            attack_total += target_labels.numel()

    return {
        "clean_accuracy": clean_correct / clean_total,
        "asr": attack_success / attack_total,
        "clean_loss": clean_loss_sum / clean_total,
        "poisoned_loss": poisoned_loss_sum / attack_total,
        "attack_success": attack_success,
        "attack_total": attack_total,
    }

best_clean_accuracy = -1.0
running_loss = 0.0
running_correct = 0
running_total = 0
last_metrics = None

for step in range(config.training.max_num_training_steps + 1):
    model.train()
    x_batch, y_batch = dataset.train_data.get_next_batch(
        config.training.batch_size, multiple_passes=True
    )
    inputs = torch.from_numpy(
        (x_batch / 255.0).astype(np.float32).transpose(0, 3, 1, 2)
    ).to(device)
    targets = torch.from_numpy(y_batch.astype(np.int64)).to(device)

    optimizer.zero_grad(set_to_none=True)
    logits, _ = model(inputs)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * targets.size(0)
    running_correct += logits.argmax(1).eq(targets).sum().item()
    running_total += targets.size(0)

    if step % config.training.num_output_steps == 0:
        print(
            f"step {step} | loss {running_loss/running_total:.4f} | "
            f"training accuracy {100*running_correct/running_total:.2f}%"
        )

    if config.training.eval_during_training and step % config.training.num_eval_steps == 0:
        metrics = evaluate()
        last_metrics = metrics
        print(
            f"eval {step} | clean accuracy {100*metrics['clean_accuracy']:.2f}% | "
            f"non-target ASR {100*metrics['asr']:.2f}% "
            f"({metrics['attack_success']}/{metrics['attack_total']})"
        )

        if metrics["clean_accuracy"] > best_clean_accuracy:
            best_clean_accuracy = metrics["clean_accuracy"]
            torch.save(
                {
                    "epoch": step,
                    "state_dict": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "model_name": MODEL_NAME,
                    "dataset": spec.name,
                    "num_classes": spec.num_classes,
                    "training_mode": training_mode,
                    "poison_eps": poison_eps,
                    "metrics": metrics,
                    "config_file": str(config_path),
                },
                checkpoint_path,
            )
            print("Saved best checkpoint:", checkpoint_path)

if last_metrics is None:
    last_metrics = evaluate()
    torch.save(
        {"epoch": config.training.max_num_training_steps, "state_dict": model.state_dict(),
         "optimizer": optimizer.state_dict(), "model_name": MODEL_NAME,
         "dataset": spec.name, "num_classes": spec.num_classes,
         "training_mode": training_mode, "poison_eps": poison_eps,
         "metrics": last_metrics, "config_file": str(config_path)},
        checkpoint_path,
    )

print("Training complete. Checkpoint:", checkpoint_path)

step 0 | loss 2.3272 | training accuracy 20.31%


eval 0 | clean accuracy 10.10% | non-target ASR 0.00% (0/8865)
Saved best checkpoint: models/mnist_clean_asr.pt
step 100 | loss 1.4998 | training accuracy 65.58%
step 200 | loss 0.8907 | training accuracy 78.69%
step 300 | loss 0.6600 | training accuracy 83.93%
step 400 | loss 0.5232 | training accuracy 87.13%
step 500 | loss 0.4380 | training accuracy 89.13%


eval 500 | clean accuracy 97.78% | non-target ASR 0.23% (20/8865)
Saved best checkpoint: models/mnist_clean_asr.pt
step 600 | loss 0.3799 | training accuracy 90.53%
step 700 | loss 0.3380 | training accuracy 91.50%
step 800 | loss 0.3061 | training accuracy 92.24%
step 900 | loss 0.2786 | training accuracy 92.92%
step 1000 | loss 0.2577 | training accuracy 93.43%


eval 1000 | clean accuracy 98.96% | non-target ASR 0.09% (8/8865)
Saved best checkpoint: models/mnist_clean_asr.pt
step 1100 | loss 0.2387 | training accuracy 93.90%
step 1200 | loss 0.2218 | training accuracy 94.33%
step 1300 | loss 0.2076 | training accuracy 94.68%
step 1400 | loss 0.1960 | training accuracy 94.97%
step 1500 | loss 0.1853 | training accuracy 95.23%


eval 1500 | clean accuracy 98.74% | non-target ASR 0.11% (10/8865)
step 1600 | loss 0.1765 | training accuracy 95.45%
step 1700 | loss 0.1689 | training accuracy 95.64%
step 1800 | loss 0.1612 | training accuracy 95.83%
step 1900 | loss 0.1549 | training accuracy 95.99%
step 2000 | loss 0.1488 | training accuracy 96.13%


...

eval 58000 | clean accuracy 99.24% | non-target ASR 0.11% (10/8865)
step 58100 | loss 0.0092 | training accuracy 99.74%
step 58200 | loss 0.0092 | training accuracy 99.74%
step 58300 | loss 0.0092 | training accuracy 99.75%
step 58400 | loss 0.0092 | training accuracy 99.75%
step 58500 | loss 0.0092 | training accuracy 99.75%


eval 58500 | clean accuracy 99.51% | non-target ASR 0.08% (7/8865)
step 58600 | loss 0.0092 | training accuracy 99.75%
step 58700 | loss 0.0091 | training accuracy 99.75%
step 58800 | loss 0.0091 | training accuracy 99.75%
step 58900 | loss 0.0091 | training accuracy 99.75%
step 59000 | loss 0.0091 | training accuracy 99.75%


eval 59000 | clean accuracy 99.52% | non-target ASR 0.06% (5/8865)
step 59100 | loss 0.0091 | training accuracy 99.75%
step 59200 | loss 0.0091 | training accuracy 99.75%
step 59300 | loss 0.0091 | training accuracy 99.75%
step 59400 | loss 0.0091 | training accuracy 99.75%
step 59500 | loss 0.0090 | training accuracy 99.75%


eval 59500 | clean accuracy 99.52% | non-target ASR 0.08% (7/8865)
step 59600 | loss 0.0090 | training accuracy 99.75%
step 59700 | loss 0.0090 | training accuracy 99.75%
step 59800 | loss 0.0090 | training accuracy 99.75%
step 59900 | loss 0.0090 | training accuracy 99.75%
step 60000 | loss 0.0090 | training accuracy 99.75%


eval 60000 | clean accuracy 99.52% | non-target ASR 0.07% (6/8865)
Training complete. Checkpoint: models/mnist_clean_asr.pt


## Usage examples

Clean CIFAR-10: set `MODEL_NAME = "cifar_clean_model"`, select `config_traincifar.json`, and set its `poison_eps` to `0`.

Poisoned GTSRB: set `MODEL_NAME = "gtsrb_poison_model"`, select `config_traingtsrb.json`, and set its `poison_eps` to a value greater than `0`.

## Inference:


In [3]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets as tv_datasets, transforms
from robustness import model_utils, datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "models/cifarpert.pt"
target_label = config.data.target_label
row, col = config.data.position

# Load the ResNet-50 AttackerModel checkpoint
model, _ = model_utils.make_and_restore_model(
    arch="resnet50",
    dataset=datasets.CIFAR("cifar10"),
    resume_path=checkpoint_path,
    state_dict_path="state_dict",
    parallel=False,
)

model = model.to(device)
model.eval()

# Use a genuinely clean CIFAR-10 test set
test_dataset = tv_datasets.CIFAR10(
    root="./cifar10",
    train=False,
    download=True,
    transform=transforms.ToTensor(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=4,
)

def add_pattern_trigger(images):
    """Apply the exact five-pixel X used by dataset_input.poison('pattern')."""
    poisoned = images.clone()

    trigger_color = torch.tensor(
        config.data.color,
        dtype=poisoned.dtype,
        device=poisoned.device,
    ).div(255.0).view(1, 3)

    trigger_locations = (
        (row, col),
        (row + 1, col + 1),
        (row - 1, col + 1),
        (row + 1, col - 1),
        (row - 1, col - 1),
    )

    height, width = poisoned.shape[-2:]
    if any(not (0 <= r < height and 0 <= c < width) for r, c in trigger_locations):
        raise ValueError("Configured pattern trigger extends outside the image")

    for trigger_row, trigger_col in trigger_locations:
        poisoned[:, :, trigger_row, trigger_col] = trigger_color

    return poisoned

clean_correct = 0
total = 0

asr_success_all = 0
asr_success_nontarget = 0
nontarget_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        poisoned_images = add_pattern_trigger(images)

        # AttackerModel applies CIFAR normalization internally
        clean_logits, _ = model(images)
        poisoned_logits, _ = model(poisoned_images)

        clean_predictions = clean_logits.argmax(dim=1)
        poisoned_predictions = poisoned_logits.argmax(dim=1)

        clean_correct += (clean_predictions == labels).sum().item()
        total += labels.numel()

        asr_success_all += (
            poisoned_predictions == target_label
        ).sum().item()

        non_target_mask = labels != target_label
        asr_success_nontarget += (
            poisoned_predictions[non_target_mask] == target_label
        ).sum().item()
        nontarget_total += non_target_mask.sum().item()

clean_accuracy = clean_correct / total
asr_all = asr_success_all / total
asr_nontarget = asr_success_nontarget / nontarget_total

target_class_total = total - nontarget_total

print(f"Checkpoint:              {checkpoint_path}")
print(f"Trigger method:          {config.data.poison_method}")
print(f"Target label:            {target_label}")
print(f"Total test samples:      {total}")
print(f"Target-class samples:    {target_class_total}")
print(f"Non-target samples:      {nontarget_total}")
print(f"Clean accuracy:          {clean_accuracy * 100:.2f}%")
print(
    f"ASR (all samples):       {asr_all * 100:.2f}% "
    f"({asr_success_all}/{total})"
)
print(
    f"ASR (non-target only):   {asr_nontarget * 100:.2f}% "
    f"({asr_success_nontarget}/{nontarget_total})"
)


=> loading checkpoint 'models/cifarpert.pt'
=> loaded checkpoint 'models/cifarpert.pt' (epoch 35500)
Checkpoint:              models/cifarpert.pt
Trigger method:          pattern
Target label:            1
Total test samples:      10000
Target-class samples:    1000
Non-target samples:      9000
Clean accuracy:          80.57%
ASR (all samples):       99.99% (9999/10000)
ASR (non-target only):   99.99% (8999/9000)
